# Code to donwload Images from .parquet

In [1]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
import asyncio
import aiohttp
from aiohttp import ClientTimeout
from PIL import Image
from io import BytesIO
from tqdm import tqdm
import warnings

In [2]:

# Ignore specific PIL warning about palette transparency
warnings.filterwarnings(
    "ignore",
    "Palette images with Transparency expressed in bytes should be converted to RGBA images",
    category=UserWarning,
)

In [ ]:
# ---------------- CONFIG ----------------
# DOWNLOAD_ROOT = r"E:\Thesis\0000_images"
# FAILED_LOG = r"E:\Thesis\0000_failed.log"
DOWNLOAD_ROOT = r"G:\Thesis\0000_images" #r"F:\Thesis\0001_images"
FAILED_LOG = r"G:\Thesis\0000_failed.log" #r"F:\Thesis\0001_failed.log"
BATCH_SIZE = 500
MAX_THREADS = 128
# ----------------------------------------

# List parquet files explicitly
# parquet_files = [
#     r"E:\Thesis\0000_embeddings\part-00000.parquet",
#     r"E:\Thesis\0000_embeddings\part-00001.parquet",
#     r"E:\Thesis\0000_embeddings\part-00002.parquet",
#     r"E:\Thesis\0000_embeddings\part-00003.parquet",
#     r"E:\Thesis\0000_embeddings\part-00004.parquet",
#     r"E:\Thesis\0000_embeddings\part-00005.parquet",
#     r"E:\Thesis\0000_embeddings\part-00006.parquet",
#     r"E:\Thesis\0000_embeddings\part-00007.parquet",
#     r"E:\Thesis\0000_embeddings\part-00008.parquet"
# ]

# parquet_files = [
#     r"F:\Thesis\0001_embeddings\part-00000.parquet",
#     r"F:\Thesis\0001_embeddings\part-00001.parquet",
#     r"F:\Thesis\0001_embeddings\part-00002.parquet",
#     r"F:\Thesis\0001_embeddings\part-00003.parquet",
#     r"F:\Thesis\0001_embeddings\part-00004.parquet",
#     r"F:\Thesis\0001_embeddings\part-00005.parquet",
#     r"F:\Thesis\0001_embeddings\part-00006.parquet",
#     r"F:\Thesis\0001_embeddings\part-00007.parquet",
#     r"F:\Thesis\0001_embeddings\part-00008.parquet"
# ]

parquet_files = [
    r"D:\ThesisFiles\test\0000_embeddings_cleaned\part-00000.parquet",
    r"D:\ThesisFiles\test\0000_embeddings_cleaned\part-00001.parquet",
    r"D:\ThesisFiles\test\0000_embeddings_cleaned\part-00002.parquet",
    r"D:\ThesisFiles\test\0000_embeddings_cleaned\part-00003.parquet",
    r"D:\ThesisFiles\test\0000_embeddings_cleaned\part-00004.parquet",
    r"D:\ThesisFiles\test\0000_embeddings_cleaned\part-00005.parquet",
    r"D:\ThesisFiles\test\0000_embeddings_cleaned\part-00006.parquet",
    r"D:\ThesisFiles\test\0000_embeddings_cleaned\part-00007.parquet",
    r"D:\ThesisFiles\test\0000_embeddings_cleaned\part-00008.parquet"
]



In [10]:
import socket
import time

def check_internet(host="8.8.8.8", port=53, timeout=3):
    """Check internet connectivity by attempting a TCP connection."""
    try:
        socket.setdefaulttimeout(timeout)
        socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect((host, port))
        return True
    except socket.error:
        return False

def wait_for_internet(max_wait=3600):
    """
    Wait until internet connection is available.
    Exponential backoff up to 1hr total wait.
    Returns True if internet is restored, False if timeout reached.
    """
    wait_time = 5  # start with 5 seconds
    total_wait = 0
    was_offline = False

    while not check_internet():
        was_offline = True
        if total_wait >= max_wait:
            print("❌ No internet connection for 1 hour. Exiting...")
            return False
        print(f"❌ No internet connection. Retrying in {wait_time} seconds...")
        time.sleep(wait_time)
        total_wait += wait_time
        wait_time = min(wait_time * 2, 3600)  # grow wait but cap at 1hr

    if was_offline:
        print("✅ Internet connection restored!")
    return True

In [ ]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
from PIL import Image
from io import BytesIO
from tqdm import tqdm
import sys
from urllib.parse import urlparse

# ---------------- CONFIG ----------------
# DOWNLOAD_ROOT = r"E:\Thesis\0000_images"
# FAILED_LOG = r"E:\Thesis\0000_failed.log"
# BATCH_SIZE = 500
# MAX_THREADS = 50  # adjust to available resources
# ----------------------------------------

# Load failures from previous runs
failed_set = set()
failed_log_path = Path(FAILED_LOG)
if failed_log_path.exists():
    with open(failed_log_path, "r") as f:
        failed_set = {line.strip() for line in f}


def download_image(idx, url):
    image_path = Path(DOWNLOAD_ROOT) / f"{idx}.jpg"
    if image_path.exists():
        return True
    try:
        if not wait_for_internet():
            sys.exit("❌ Fatal Error: No internet connection for 1 hour. Job terminated.")

        parsed = urlparse(url)
        referer = f"{parsed.scheme}://{parsed.hostname}/"

        headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/123.0.0.0 Safari/537.36"
            ),
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.9",
            "Referer": referer,
            "DNT": "1",  # Do Not Track
            "Connection": "keep-alive",
        }

        with requests.Session() as session:  # auto-close session
            # resp = session.get(url, timeout=15, headers=headers)
            resp = session.get(url, timeout=5, headers=headers)
            # resp = requests.get(url, timeout=15, headers=headers)
            resp.raise_for_status()
            image = Image.open(BytesIO(resp.content)).convert("RGB")
            image.save(image_path)
            return True
    except Exception:
        with open(FAILED_LOG, "a") as f:
            f.write(f"{idx}\n")
            f.flush()
        return False


def process_parquet(parquet_path):
    download_dir = Path(DOWNLOAD_ROOT)
    download_dir.mkdir(parents=True, exist_ok=True)
    existing_images = {p.stem for p in download_dir.glob("*.jpg") if p.stem.isdigit()}

    parquet = pq.ParquetFile(parquet_path)
    total_to_download = 0

    # First count total images
    for rg in range(parquet.num_row_groups):
        df = parquet.read_row_group(rg).to_pandas()
        df = df[df["embeddings_result"].notnull()]
        for _, row in df.iterrows():
            idx = str(row["original_image_index"])
            if idx not in existing_images and idx not in failed_set:
                total_to_download += 1

    print(f"{parquet_path}: {total_to_download} images to download.")

    with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
        with tqdm(total=total_to_download, desc=Path(parquet_path).stem) as pbar:
            futures = []
            for rg in range(parquet.num_row_groups):
                df = parquet.read_row_group(rg).to_pandas()
                df = df[df["embeddings_result"].notnull()]

                batch = []
                for _, row in df.iterrows():
                    idx = str(row["original_image_index"])
                    if idx not in existing_images and idx not in failed_set:
                        batch.append((idx, row["url"]))

                    if len(batch) >= BATCH_SIZE:

                        # Wait for internet before firing batch
                        if not wait_for_internet():
                           sys.exit("❌ Fatal Error: No internet connection for 1 hour. Job terminated.")

                        for idx_url in batch:
                            futures.append(executor.submit(download_image, *idx_url))
                        batch = []

                        # Update progress as futures complete
                        for future in as_completed(futures):
                            pbar.update(1)
                        futures = []

                # Remaining batch
                if batch:
                    if not wait_for_internet():
                        sys.exit("❌ Fatal Error: No internet connection for 1 hour. Job terminated.")

                    for idx_url in batch:
                        futures.append(executor.submit(download_image, *idx_url))
                    for future in as_completed(futures):
                        pbar.update(1)
                    futures = []


def main(parquet_files):
    for parquet_path in parquet_files:
        print(f"Processing {parquet_path}")
        process_parquet(parquet_path)
    print("All downloads completed.")


In [12]:
import aiofiles
await main(parquet_files)

Processing F:\Thesis\0001_embeddings\part-00000.parquet
F:\Thesis\0001_embeddings\part-00000.parquet: 0 images to download.


part-00000: 0it [01:03, ?it/s]


Processing F:\Thesis\0001_embeddings\part-00001.parquet
F:\Thesis\0001_embeddings\part-00001.parquet: 0 images to download.


part-00001: 0it [01:02, ?it/s]


Processing F:\Thesis\0001_embeddings\part-00002.parquet
F:\Thesis\0001_embeddings\part-00002.parquet: 0 images to download.


...

part-00007: 100%|██████████| 4/4 [00:54<00:00, 13.62s/it]


Processing F:\Thesis\0001_embeddings\part-00008.parquet
F:\Thesis\0001_embeddings\part-00008.parquet: 0 images to download.


part-00008: 0it [00:09, ?it/s]


All downloads completed.


TypeError: object NoneType can't be used in 'await' expression

In [3]:
import pandas as pd

df = pd.read_parquet(r"D:\ThesisFiles\test\0003_embeddings_cleaned\part-00000.parquet")

df.head()

,original_image_index,url,caption,embeddings_result,similarity,parquet_file_name
0,0,https://bt-photos.global.ssl.fastly.net/kaar/6...,"321 Vista View Way, Maryville, TN 37801 (#1084...","[0.019070197, 0.021310333, 0.020855963, -0.018...",None,0003.parquet
1,1,https://www.specsserver.com/CACHE/FRLOPRMTKKDU...,"36'' 6-Burner Dual Fuel Freestanding Range, Co...",None,None,0003.parquet
2,2,https://i.pinimg.com/736x/e3/26/76/e326768f576...,Fabelab AW 17 Collection http://petitandsmall....,"[0.02106217, 0.028399475, 0.013184387, 0.04851...",None,0003.parquet
3,3,https://www.freshsoundrecords.com/9478-medium_...,"Something New, Something Blue + Swinging 'Guys...","[0.027745398, 0.04687661, 0.042362824, -0.0035...",None,0003.parquet
4,4,https://i2.wp.com/d.gr-assets.com/books/132787...,Beautiful Creatures,"[-0.00017633525, 0.025518645, -0.01657377, -0....",None,0003.parquet


In [8]:
full_url = df[df['original_image_index'] == 6]['url'].iloc[0]
print(full_url)

https://blog.spoongraphics.co.uk/wp-content/uploads/2008/01/iron_man_ver2.jpg
